# Submissão 2 - Abordagem Híbrida: Extrator de Embeddings (BERT) + Classificador (DNN PyTorch)

Neste notebook implementamos a pipeline hierárquica usando Embeddings pré-treinados:
1. Extraímos os Text Embeddings usando o BERT (sem treino).
2. **Modelo Binário (DNN)**: Distingue Humanos vs IA usando os embeddings.
3. **Modelo Multiclasse (DNN)**: Distingue as classes de IA (Google, Meta, Mistral, OpenAI) apenas para os exemplos IA.

In [ ]:

!pip install transformers -q
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
from transformers import DistilBertModel, DistilBertTokenizerFast
import os

DATA_DIR = "../data"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

df_train = pd.read_csv(os.path.join(DATA_DIR, "dataset_treino.csv"))
df_val = pd.read_csv(os.path.join(DATA_DIR, "dataset_validacao.csv"))

def clean_text(text):
    import re
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

for df in [df_train, df_val]:
    df['Text'] = df['Text'].apply(clean_text)

df_train['binary_label'] = df_train['source_name'].apply(lambda x: 0 if x.lower() == 'human' else 1)
df_val['binary_label'] = df_val['source_name'].apply(lambda x: 0 if x.lower() == 'human' else 1)

unique_labels = df_train['source_name'].dropna().unique().tolist()
ai_classes = sorted([c for c in unique_labels if c.lower() != 'human'])
ai_mapping = {k: v for v, k in enumerate(ai_classes)}
reverse_ai_mapping = {v: k for k, v in ai_mapping.items()}
print("Mapeamento Apenas IA:", ai_mapping)

df_train_ai = df_train[df_train['binary_label'] == 1].copy()
df_val_ai = df_val[df_val['binary_label'] == 1].copy()

df_train_ai['ai_label'] = df_train_ai['source_name'].map(ai_mapping)
df_val_ai['ai_label'] = df_val_ai['source_name'].map(ai_mapping)


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
bert_model.eval()

def get_bert_embeddings(texts, batch_size=64):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size].tolist()
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
            out = bert_model(**enc).last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1).expand(out.size()).float()
            sum_embeddings = torch.sum(out * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            mean_pooled = sum_embeddings / sum_mask
            embeddings.append(mean_pooled.cpu())
    return torch.cat(embeddings, dim=0)

print("A extrair embeddings BERT para o Treino (Binário e Multi)...")
X_train_bin = get_bert_embeddings(df_train['Text'])
X_train_multi = get_bert_embeddings(df_train_ai['Text'])

print("A extrair embeddings BERT para Validação...")
X_val_bin = get_bert_embeddings(df_val['Text'])
X_val_multi = get_bert_embeddings(df_val_ai['Text'])


In [ ]:

class DNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(DNClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

import torch.optim as optim
y_train_bin = torch.tensor(df_train['binary_label'].values, dtype=torch.long)
train_bin_loader = DataLoader(TensorDataset(X_train_bin, y_train_bin), batch_size=64, shuffle=True)

bin_model = DNClassifier(768, 128, 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_bin = optim.Adam(bin_model.parameters(), lr=1e-3)

print("A treinar Modelo DNN Binário")
for epoch in range(15):
    bin_model.train()
    total_loss = 0
    for X_b, y_b in train_bin_loader:
        optimizer_bin.zero_grad()
        out = bin_model(X_b.to(device))
        loss = criterion(out, y_b.to(device))
        loss.backward()
        optimizer_bin.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Época {epoch+1} - Loss: {total_loss/len(train_bin_loader):.4f}")

torch.save(bin_model.state_dict(), os.path.join(MODELS_DIR, 'hybrid_binary_dnn.pt'))


In [ ]:

y_train_multi = torch.tensor(df_train_ai['ai_label'].values, dtype=torch.long)
train_multi_loader = DataLoader(TensorDataset(X_train_multi, y_train_multi), batch_size=64, shuffle=True)

multi_model = DNClassifier(768, 128, len(ai_classes)).to(device)
optimizer_multi = optim.Adam(multi_model.parameters(), lr=1e-3)

print("\nA treinar Modelo DNN Multiclasse")
for epoch in range(30):
    multi_model.train()
    total_loss = 0
    for X_b, y_b in train_multi_loader:
        optimizer_multi.zero_grad()
        out = multi_model(X_b.to(device))
        loss = criterion(out, y_b.to(device))
        loss.backward()
        optimizer_multi.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Época {epoch+1} - Loss: {total_loss/len(train_multi_loader):.4f}")

torch.save(multi_model.state_dict(), os.path.join(MODELS_DIR, 'hybrid_multi_dnn.pt'))


In [ ]:

bin_model.eval()
multi_model.eval()

predictions = []

with torch.no_grad():
    bin_out = bin_model(X_val_bin.to(device))
    bin_preds = torch.argmax(bin_out, dim=1).cpu().numpy()
    
    for i, is_ai in enumerate(bin_preds):
        if is_ai == 0:
            predictions.append('human')
        else:
            emb = X_val_bin[i].unsqueeze(0).to(device)
            multi_out = multi_model(emb)
            multi_pred = torch.argmax(multi_out, dim=1).cpu().item()
            predictions.append(reverse_ai_mapping[multi_pred])

print("\nRelatório Final (Híbrido Hierárquico):")
print(classification_report(df_val['source_name'], predictions))
print("Accuracy Final:", accuracy_score(df_val['source_name'], predictions))

df_submission = df_val.copy()
df_submission['predicted_source'] = predictions
csv_out = '../subm2/subm_hybrid_hierarquico.csv'
os.makedirs('../subm2', exist_ok=True)
df_submission.to_csv(csv_out, index=False)
print(f"\nSubmissão salva em {csv_out}")
